In [14]:
import pandas as pd
import numpy as np
import requests
import datetime
import time
import os
import json
from datetime import datetime




In [7]:
url="https://steamspy.com/api.php"
PAGES=[0,1,2]
time_delay = 60
RESET = 3
timeout = 30

In [8]:
def get_one_page(page, retries=RESET):
    """Качает одну страницу request=all. Возвращает словарь с играми."""
    params = {
        "request": "all",
        "page": page
    }
    
    for attempt in range(retries):
        try:
            response = requests.get(url, params=params, timeout=timeout)
            response.raise_for_status()
            return response.json()
        except (requests.RequestException, ValueError) as e:
            print(f"  Ошибка на странице {page} (попытка {attempt + 1}): {e}")
            if attempt < retries - 1:
                time.sleep(5)
    return {}

In [9]:
def get_many_pages(pages):
    """Качает несколько страниц подряд с паузой между запросами."""
    all_games = {}
    
    for i, page in enumerate(pages):
        print(f"Загружаю страницу {page}...")
        page_data = get_one_page(page)
        print(f"  Получено игр: {len(page_data)}")
        all_games.update(page_data)
        
        # Пауза между запросами (кроме последнего)
        if i < len(pages) - 1:
            print(f"  Жду {time_delay} секунд...")
            time.sleep(time_delay)
    
    return all_games

In [10]:
games = get_many_pages(PAGES)
print(f"\nВсего игр собрано: {len(games)}")

Загружаю страницу 0...
  Получено игр: 1000
  Жду 60 секунд...
Загружаю страницу 1...
  Получено игр: 1000
  Жду 60 секунд...
Загружаю страницу 2...
  Получено игр: 1000

Всего игр собрано: 3000


In [11]:
print("Топ-5 игр по владельцам:\n")
for i, (appid, game) in enumerate(games.items()):
    if i >= 5:
        break
    print(f"{i+1}. {game.get('name')} — владельцы: {game.get('owners')}")

Топ-5 игр по владельцам:

1. Counter-Strike: Global Offensive — владельцы: 100,000,000 .. 200,000,000
2. Apex Legends — владельцы: 100,000,000 .. 200,000,000
3. PUBG: BATTLEGROUNDS — владельцы: 100,000,000 .. 200,000,000
4. Palworld — владельцы: 50,000,000 .. 100,000,000
5. Team Fortress 2 — владельцы: 50,000,000 .. 100,000,000


In [15]:
update_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"update_at: {update_at}")

update_at: 2026-10-06 01:25:50


In [16]:
df = pd.DataFrame(games.values())
print(df.shape)
df.head()

(3000, 17)


,appid,name,developer,publisher,score_rank,positive,negative,userscore,owners,average_forever,average_2weeks,median_forever,median_2weeks,price,initialprice,discount,ccu
0,730,Counter-Strike: Global Offensive,Valve,Valve,,7642084,1173003,0,"100,000,000 .. 200,000,000",0,0,0,0,0,0,0,1013936
1,1172470,Apex Legends,Respawn,Electronic Arts,,668053,326926,0,"100,000,000 .. 200,000,000",0,0,0,0,0,0,0,124262
2,578080,PUBG: BATTLEGROUNDS,PUBG Corporation,"KRAFTON, Inc.",,1520457,1037487,0,"100,000,000 .. 200,000,000",0,0,0,0,0,0,0,314682
3,1623730,Palworld,Pocketpair,Pocketpair,,358266,22443,0,"50,000,000 .. 100,000,000",0,0,0,0,2099,2999,30,18028
4,440,Team Fortress 2,Valve,Valve,,1044264,117208,0,"50,000,000 .. 100,000,000",0,0,0,0,0,0,0,43819


In [17]:
df["update_at"] = update_at
df["business_date"] = None   # в API Steam Spy бизнес-даты нет

In [18]:
os.makedirs("tmp", exist_ok=True)
df.to_csv("tmp/steamspy_pages_0_2.csv", index=False, encoding="utf-8-sig")
print("Сохранено: tmp/steamspy_pages_0_2.csv")

Сохранено: tmp/steamspy_pages_0_2.csv


In [20]:
check = pd.read_csv("tmp/steamspy_pages_0_2.csv")
print(check.shape)
check.head()

(3000, 19)


,appid,name,developer,publisher,score_rank,positive,negative,userscore,owners,average_forever,average_2weeks,median_forever,median_2weeks,price,initialprice,discount,ccu,update_at,business_date
0,730,Counter-Strike: Global Offensive,Valve,Valve,NaN,7642084,1173003,0,"100,000,000 .. 200,000,000",0,0,0,0,0,0,0,1013936,2026-10-06 01:25:50,NaN
1,1172470,Apex Legends,Respawn,Electronic Arts,NaN,668053,326926,0,"100,000,000 .. 200,000,000",0,0,0,0,0,0,0,124262,2026-10-06 01:25:50,NaN
2,578080,PUBG: BATTLEGROUNDS,PUBG Corporation,"KRAFTON, Inc.",NaN,1520457,1037487,0,"100,000,000 .. 200,000,000",0,0,0,0,0,0,0,314682,2026-10-06 01:25:50,NaN
3,1623730,Palworld,Pocketpair,Pocketpair,NaN,358266,22443,0,"50,000,000 .. 100,000,000",0,0,0,0,2099,2999,30,18028,2026-10-06 01:25:50,NaN
4,440,Team Fortress 2,Valve,Valve,NaN,1044264,117208,0,"50,000,000 .. 100,000,000",0,0,0,0,0,0,0,43819,2026-10-06 01:25:50,NaN
